# 02 - Data Cleaning

**BlueStock MF Capstone** — Clean, validate, and standardise all 10 datasets.
Exports cleaned CSVs to `data/processed/` and loads into SQLite DB.

---

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
from pathlib import Path

ROOT = Path(os.path.abspath('')).parent if 'notebooks' in os.path.abspath('') else Path(os.path.abspath(''))
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
DB_PATH = ROOT / 'data' / 'db' / 'bluestock_mf.db'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / 'data' / 'db').mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')

---
## 1. Clean NAV History

**Key steps:**
- Parse dates, sort by fund + date
- Forward-fill missing NAVs (weekends/holidays)
- Remove duplicate rows and invalid NAVs

In [ ]:
df_nav = pd.read_csv(RAW_DIR / '02_nav_history.csv')
print(f'Raw NAV: {df_nav.shape}')

df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(['amfi_code', 'date']).drop_duplicates()

# Forward-fill NAV for weekends/holidays within each fund
df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()

# Remove invalid NAVs
invalid = df_nav[df_nav['nav'] <= 0]
print(f'Invalid NAV rows (<=0): {len(invalid)}')
df_nav = df_nav[df_nav['nav'] > 0]

print(f'Cleaned NAV: {df_nav.shape}')
print(f'Date range: {df_nav["date"].min().date()} to {df_nav["date"].max().date()}')
print(f'Null NAVs remaining: {df_nav["nav"].isnull().sum()}')

df_nav.to_csv(PROCESSED_DIR / 'clean_nav_history.csv', index=False)
print('Saved: clean_nav_history.csv')

---
## 2. Clean Investor Transactions

**Key steps:**
- Parse dates, standardise transaction_type (SIP/Lumpsum/Redemption)
- Remove zero/negative amounts
- Validate KYC status

In [ ]:
df_tx = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')
print(f'Raw transactions: {df_tx.shape}')

df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Standardise transaction types
type_map = {
    'sip': 'SIP', 'SIP': 'SIP',
    'lumpsum': 'Lumpsum', 'Lumpsum': 'Lumpsum',
    'redemption': 'Redemption', 'Redemption': 'Redemption',
}
df_tx['transaction_type'] = df_tx['transaction_type'].astype(str).str.strip().map(type_map)

# Remove non-positive amounts
before = len(df_tx)
df_tx = df_tx[df_tx['amount_inr'] > 0]
print(f'Removed {before - len(df_tx)} non-positive amount rows')

# KYC validation
print(f'\nKYC status distribution:')
display(df_tx['kyc_status'].value_counts())

print(f'\nTransaction type distribution:')
display(df_tx['transaction_type'].value_counts())

print(f'\nCleaned transactions: {df_tx.shape}')
df_tx.to_csv(PROCESSED_DIR / 'clean_transactions.csv', index=False)
print('Saved: clean_transactions.csv')

---
## 3. Clean Scheme Performance

**Key steps:**
- Coerce numeric columns (handle any non-numeric strings)
- Validate expense ratio ranges
- Check for anomalous Sharpe/Sortino values

In [ ]:
df_perf = pd.read_csv(RAW_DIR / '07_scheme_performance.csv')
print(f'Raw performance: {df_perf.shape}')

numeric_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
                'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
                'expense_ratio_pct', 'max_drawdown_pct']

for col in numeric_cols:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# Flag expense ratio anomalies
anomalies = df_perf[(df_perf['expense_ratio_pct'] < 0.1) | (df_perf['expense_ratio_pct'] > 2.5)]
print(f'\nExpense ratio anomalies ({len(anomalies)}):')
if len(anomalies) > 0:
    display(anomalies[['scheme_name', 'expense_ratio_pct']])
else:
    print('  None found')

print(f'\nNull counts in numeric columns:')
print(df_perf[numeric_cols].isnull().sum().to_string())

df_perf.to_csv(PROCESSED_DIR / 'clean_scheme_performance.csv', index=False)
print('\nSaved: clean_scheme_performance.csv')

---
## 4. Clean Remaining Datasets (7 files)

Basic cleaning: drop duplicates, parse dates where applicable, export to processed/.

In [ ]:
remaining = {
    '01_fund_master.csv':                                'clean_fund_master.csv',
    '03_aum_by_fund_house.csv':                          'clean_aum_by_fund_house.csv',
    '04_monthly_sip_inflows.csv':                        'clean_monthly_sip_inflows.csv',
    '05_category_inflows.csv':                           'clean_category_inflows.csv',
    '06_industry_folio_count.csv':                       'clean_industry_folio_count.csv',
    '09_portfolio_holdings - 09_portfolio_holdings.csv':  'clean_portfolio_holdings.csv',
    '10_benchmark_indices - 10_benchmark_indices.csv':    'clean_benchmark_indices.csv',
}

for raw_name, clean_name in remaining.items():
    df = pd.read_csv(RAW_DIR / raw_name)
    before = len(df)
    df = df.drop_duplicates()
    dropped = before - len(df)
    df.to_csv(PROCESSED_DIR / clean_name, index=False)
    print(f'{clean_name:40s} | {len(df):>6} rows | {dropped} dupes removed')

---
## 5. Load into SQLite Database

Create `data/db/bluestock_mf.db` using schema from `sql/schema.sql`.

In [ ]:
# Apply schema
with sqlite3.connect(DB_PATH) as conn:
    with open(ROOT / 'sql' / 'schema.sql', 'r') as f:
        conn.executescript(f.read())
    print('Schema applied.')

    # Load tables
    df = pd.read_csv(PROCESSED_DIR / 'clean_fund_master.csv')
    cols = ['amfi_code','fund_house','scheme_name','category','sub_category','expense_ratio_pct','risk_category']
    cols = [c for c in cols if c in df.columns]
    df[cols].to_sql('dim_fund', conn, if_exists='replace', index=False)
    print(f'dim_fund:           {len(df)} rows')

    df = pd.read_csv(PROCESSED_DIR / 'clean_nav_history.csv').rename(columns={'date':'nav_date'})
    df[['amfi_code','nav_date','nav']].to_sql('fact_nav', conn, if_exists='replace', index=False)
    print(f'fact_nav:           {len(df)} rows')

    df = pd.read_csv(PROCESSED_DIR / 'clean_transactions.csv')
    df.to_sql('fact_transactions', conn, if_exists='replace', index=False)
    print(f'fact_transactions:  {len(df)} rows')

    df = pd.read_csv(PROCESSED_DIR / 'clean_scheme_performance.csv')
    pcols = ['amfi_code','return_1yr_pct','return_3yr_pct','return_5yr_pct','alpha','beta','sharpe_ratio','sortino_ratio','max_drawdown_pct']
    pcols = [c for c in pcols if c in df.columns]
    df[pcols].to_sql('fact_performance', conn, if_exists='replace', index=False)
    print(f'fact_performance:   {len(df)} rows')

    df = pd.read_csv(PROCESSED_DIR / 'clean_aum_by_fund_house.csv')
    df['quarter'] = pd.to_datetime(df['date']).dt.to_period('Q').astype(str)
    df[['fund_house','quarter','aum_crore']].to_sql('fact_aum', conn, if_exists='replace', index=False)
    print(f'fact_aum:           {len(df)} rows')

    df = pd.read_csv(PROCESSED_DIR / 'clean_monthly_sip_inflows.csv')
    df.to_sql('sip_inflows', conn, if_exists='replace', index=False)
    print(f'sip_inflows:        {len(df)} rows')

# Verify
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print(f'\nTables in DB: {list(tables["name"])}')

import os
print(f'DB size: {os.path.getsize(DB_PATH)/1e6:.1f} MB')

---
## 6. Verification — Sample SQL Queries

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    # Row counts
    for table in ['dim_fund', 'fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum', 'sip_inflows']:
        count = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {table}', conn)['cnt'][0]
        print(f'  {table:25s}: {count:>8,} rows')

    # Sample query: Top 5 funds by Sharpe
    print('\n  Top 5 Funds by Sharpe Ratio:')
    result = pd.read_sql("""
        SELECT f.scheme_name, p.sharpe_ratio, p.return_1yr_pct
        FROM fact_performance p
        JOIN dim_fund f ON p.amfi_code = f.amfi_code
        ORDER BY p.sharpe_ratio DESC
        LIMIT 5
    """, conn)
    display(result)

---
## Summary

- **10 datasets cleaned** and saved to `data/processed/`
- **SQLite database** created at `data/db/bluestock_mf.db` with 6 tables
- **Key cleaning operations:**
  - NAV: date parsing, ffill for weekends/holidays, invalid NAV removal
  - Transactions: type standardisation, positive-amount filter, KYC validation
  - Performance: numeric coercion, expense ratio anomaly detection
- Data is ready for EDA in notebook `03_eda_analysis.ipynb`